# Live exercise: joins, worked solutions

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

# Part 1

## 1. Total minutes per country

25 countries. Germany, the Netherlands and Norway at the top.

In [ ]:
run("""
SELECT a.country,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.country
ORDER BY minutes DESC
""")

## 2. Plays per genre, with the average

Jazz has by far the longest average play at 6.57 minutes, despite having the
second fewest plays. Total and average disagree, as usual.

In [ ]:
run("""
SELECT a.genre,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes,
       ROUND(AVG(p.minutes_played), 2) AS avg_min
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.genre
ORDER BY plays DESC
""")

## 3. Every artist, including the never-played ones

Two things make this correct: `LEFT JOIN` keeps the artists with no plays,
and `COUNT(p.play_id)` rather than `COUNT(*)` reports them as 0.

In [ ]:
run("""
SELECT a.artist_name,
       a.genre,
       COUNT(p.play_id)                AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM artists a
LEFT JOIN plays p USING (artist_id)
GROUP BY a.artist_id
ORDER BY plays
LIMIT 5
""")

## 4. Top three artists in 2025, by minutes

The year filter is a `WHERE`, because it removes rows and has nothing to do
with the grouping. Filtering rows is cheaper than filtering groups, so do it
there whenever you can.

In [ ]:
run("""
SELECT a.artist_name,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
WHERE p.played_at >= '2025-01-01'
GROUP BY a.artist_id
ORDER BY minutes DESC
LIMIT 3
""")

# Part 2

## 5. The same query with `awards` joined on

In [ ]:
run("""
SELECT a.country,
       COUNT(*)                        AS rows_now,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
JOIN awards w ON w.artist_id = p.artist_id
GROUP BY a.country
ORDER BY minutes DESC
""")

## 6. What happened

| | before | after |
|---|---|---|
| top country | Germany 1285.1 | **Norway 2538.5** |
| Norway | 1184.0 | 2538.5 |
| countries listed | **25** | **14** |
| play rows involved | 2,183 | 2,371 |

Norway more than doubled and jumped to the top. Eleven countries vanished
entirely.

## 7. Why

**The inflation.** The `awards` table has one row per award, and an artist
can have several. Fjord & Flint, from Norway, has 3 awards and 237 plays.
The join matches every play against every award for that artist, so those
237 plays become 711 rows and their minutes are counted three times over.

Norway went to the top because its most-played artist happens to have three
awards. The number is not measuring listening any more. It is measuring
listening multiplied by awards, which is not a quantity anybody wanted.

**The disappearance.** This is the easier one to miss. The join to `awards`
is an inner join, so any artist with no awards is dropped, and with them
every country whose artists have never won anything. Two bugs in one query:
inflated numbers **and** missing rows.

Here is the second bug on its own, counted:

In [ ]:
run("SELECT COUNT(DISTINCT country) AS countries_in_artists FROM artists")

In [ ]:
run("""
SELECT COUNT(DISTINCT a.country) AS countries_after_awards_join
FROM plays p
JOIN artists a USING (artist_id)
JOIN awards w ON w.artist_id = p.artist_id
""")

## 8. The fix

Aggregate each table down to one row per artist **first**, then join. After
that the awards join cannot duplicate a play, because there are no play rows
left to duplicate.

In [ ]:
run("""
WITH per_artist AS (
    SELECT artist_id,
           COUNT(*)            AS plays,
           SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
awards_per_artist AS (
    SELECT artist_id, COUNT(*) AS awards
    FROM awards
    GROUP BY artist_id
)
SELECT a.country,
       SUM(s.plays)                AS plays,
       ROUND(SUM(s.minutes), 1)    AS minutes,
       COALESCE(SUM(aw.awards), 0) AS awards
FROM per_artist s
JOIN artists a USING (artist_id)
LEFT JOIN awards_per_artist aw USING (artist_id)
GROUP BY a.country
ORDER BY minutes DESC
""")

Germany 1285.1, Netherlands 1265.2, Norway 1184.0: back to the query 1
numbers exactly, all 25 countries present, and now with award counts
alongside.

Two details worth pointing at:

* **Two CTEs**, each reducing a table to one row per artist before anything
  is joined. That is the general recipe: get both sides down to the same
  grain, then join.

* `COALESCE(x, 0)` means "x, or 0 if x is NULL". Countries with no awards
  would otherwise show `NULL` rather than `0`, which is accurate and reads
  badly in a report.

In [ ]:
con.close()
print("done")